# 📊 Notebook 01 — Carga y Limpieza de Datos
## **Proyecto: Análisis de Rendimiento Liga BetPlay 2025**
### Equipo 3 — Bootcamp AVD Talento Tech

**Objetivo general:** Cargar el dataset crudo, identificar y corregir
problemas de calidad de datos antes del análisis exploratorio (EDA).

**Problemas que vamos a resolver en este notebook:**

| # | Problema | Técnica |
|---|---|---|
| 1 | Duplicados exactos y con variantes de nombre | `drop_duplicates()` |
| 2 | Valores nulos en columnas numéricas | `fillna()` / `dropna()` |
| 3 | Tipos de datos incorrectos | `astype()` |
| 4 | Nombres de clubes con texto extra | `str.replace()` |

> ⚠️ **Regla de oro:** nunca modificar el archivo crudo original.
> Siempre trabajamos sobre una copia y guardamos el resultado limpio
> en la carpeta `02_datos/procesados/`.

## **📦 Paso 1 — Importar librerías**

Antes de trabajar con datos necesitamos cargar las herramientas:

- **pandas:** manejo de tablas (DataFrames)
- **numpy:** operaciones numéricas y manejo de nulos (`np.nan`)

> 💡 En Python, importar una librería es como abrir una caja de
> herramientas: no consume tiempo de cómputo, solo deja las
> funciones disponibles para usarlas después.

In [12]:
# Librerías utilizadas para limpieza
import numpy as np
import pandas as pd

## **📂 Paso 2 — Conectar Drive y cargar el dataset**

Usamos `drive.mount()` para conectar Google Colab con nuestra
carpeta en Google Drive. Sin este paso, Colab no puede ver
los archivos que tenemos guardados.

`pd.read_csv()` lee el archivo y lo convierte en un DataFrame:
una tabla con filas y columnas que pandas puede manipular.

**Ruta del archivo crudo:**
`Proyecto_AVD_Equipo_N3 → 02_datos → 01_Crudos → betplay_2025_crudo.csv`

> 💡 Siempre verificamos cuántas filas cargó para confirmar
> que el archivo se leyó completo. Esperamos **23 filas**
> (20 originales + 3 duplicados intencionales).

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

ruta = '/content/drive/MyDrive/PROYECTO FINAL/Proyecto-AVD-Equipo-3/02_datos/01_Crudos/betplay_2025_crudo.csv'

df = pd.read_csv(ruta)
print(f"Filas cargadas: {len(df)}")
print(f"Columnas: {list(df.columns)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Filas cargadas: 23
Columnas: ['Posicion', 'Club', 'Partidos_jugados', 'Ganados', 'Empatados', 'Perdidos', 'Goles_favor', 'Goles_en_contra', 'Puntos', 'Diferencia_gol']


## **🔍 Paso 3 — Diagnóstico de calidad**

Antes de limpiar, necesitamos entender qué tan "sucio" está
el dataset. Este paso responde 4 preguntas clave:

| Pregunta | Comando pandas |
|---|---|
| ¿Cuántas filas y columnas hay? | `df.shape` |
| ¿Los tipos de datos son correctos? | `df.dtypes` |
| ¿Cuántos valores nulos hay y dónde? | `df.isnull().sum()` |
| ¿Hay filas completamente repetidas? | `df.duplicated().sum()` |

**Resultados esperados:**
- Filas: 23 (hay duplicados por detectar)
- Nulos: 4 celdas en columnas distintas
- Duplicados exactos: al menos 2

> ⚠️ Un duplicado no siempre es una fila idéntica. A veces el
> mismo club aparece con nombres ligeramente distintos
> (ej: "Millonarios FC" vs "Millonarios F.C."). Esos los
> detectamos en el siguiente paso.

In [21]:
# ── Diagnóstico general del dataset ──────────────────────
print("=== FORMA DEL DATASET ===")
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")

print("\n=== TIPOS DE DATOS ===")
print(df.dtypes)

print("\n=== VALORES NULOS POR COLUMNA ===")
print(df.isnull().sum())

print("\n=== DUPLICADOS EXACTOS ===")
print(f"Filas duplicadas: {df.duplicated().sum()}")

print("\n=== PRIMERAS 5 FILAS ===")
print(df.head(5))

=== FORMA DEL DATASET ===
Filas: 23 | Columnas: 10

=== TIPOS DE DATOS ===
Posicion              int64
Club                 object
Partidos_jugados      int64
Ganados             float64
Empatados             int64
Perdidos              int64
Goles_favor         float64
Goles_en_contra       int64
Puntos              float64
Diferencia_gol      float64
dtype: object

=== VALORES NULOS POR COLUMNA ===
Posicion            0
Club                0
Partidos_jugados    0
Ganados             1
Empatados           0
Perdidos            0
Goles_favor         1
Goles_en_contra     0
Puntos              1
Diferencia_gol      1
dtype: int64

=== DUPLICADOS EXACTOS ===
Filas duplicadas: 1

=== PRIMERAS 5 FILAS ===
   Posicion                    Club  Partidos_jugados  Ganados  Empatados  \
0        16             Envigado FC                20      4.0          8   
1        10         Águilas Doradas                20      7.0          6   
2         1  Independiente Medellín                20     

## **🔄 Paso 4 — Detección y corrección de duplicados**

Existen **dos tipos de duplicados** que debemos corregir por separado:

**Tipo 1 — Duplicado exacto:** la misma fila repetida completamente.
Se detecta y elimina con `drop_duplicates()`.

**Tipo 2 — Duplicado por nombre:** el mismo club con escritura
diferente (ej: "Millonarios FC" vs "Millonarios F.C.").
Este NO lo detecta `drop_duplicates()` — hay que buscarlo
inspeccionando los nombres manualmente o con `str.replace()`.

**¿Por qué es importante eliminarlos?**
Si un equipo aparece dos veces en el análisis, los promedios
y rankings quedan distorsionados. Un club podría aparecer
en el top 5 y en el fondo de la tabla al mismo tiempo.

> 💡 Siempre verificamos cuántas filas quedan **antes y después**
> de eliminar duplicados para confirmar que el proceso funcionó.

In [22]:
# ── Paso 4: Corrección de duplicados ─────────────────────

print(f"Filas antes de limpiar duplicados: {len(df)}")

# Tipo 1: duplicados exactos
df = df.drop_duplicates()
print(f"Filas después de drop_duplicates(): {len(df)}")

# Tipo 2: variantes de nombre — estandarizar
df["Club"] = df["Club"].str.replace(r"\.", "", regex=True).str.strip()
df["Club"] = df["Club"].str.replace(r"\s*-\s*Campeón.*$", "", regex=True).str.strip()

# Eliminar duplicados que quedaron tras estandarizar nombres
df = df.drop_duplicates(subset="Club")
print(f"Filas después de limpiar variantes de nombre: {len(df)}")

# Verificar clubs únicos
print(f"\nClubes únicos: {df['Club'].nunique()}")
print(df["Club"].sort_values().tolist())

Filas antes de limpiar duplicados: 23
Filas después de drop_duplicates(): 22
Filas después de limpiar variantes de nombre: 20

Clubes únicos: 20
['Alianza FC', 'Asociación Deportivo Pasto', 'Atlético Bucaramanga', 'Atlético Nacional', 'Boyacá Chicó FC', 'CD América de Cali', 'Deportes Tolima', 'Deportivo Cali', 'Deportivo Pereira', 'Envigado FC', 'Fortaleza CEIF', 'Independiente Medellín', 'Independiente Santa Fe', 'Internacional de Bogotá', 'Junior de Barranquilla', 'Llaneros FC', 'Millonarios FC', 'Once Caldas', 'Unión Magdalena', 'Águilas Doradas']


## **🕳️ Paso 5 — Detección y corrección de valores nulos**

Un **valor nulo** (`NaN`) es una celda vacía: el dato existe
pero no fue registrado o se perdió en la carga.

**¿Por qué son un problema?**
- Las fórmulas matemáticas fallan o dan resultados incorrectos
- Los gráficos pueden omitir equipos completos
- El ranking de descenso quedaría incompleto

**Estrategias para tratar nulos:**

| Estrategia | Cuándo usarla |
|---|---|
| `fillna(mediana)` | Columnas numéricas con pocos nulos |
| `fillna(0)` | Cuando el nulo realmente significa cero |
| `dropna()` | Cuando la fila entera no tiene sentido sin ese dato |

> 💡 En este dataset usamos la **mediana** (no el promedio)
> porque es menos sensible a valores extremos. Si un equipo
> tiene 40 puntos y otro tiene 14, el promedio se distorsiona;
> la mediana no.

In [23]:
# ── Paso 5: Corrección de valores nulos ──────────────────

print("=== NULOS ANTES DE CORREGIR ===")
print(df.isnull().sum())

# Columnas numéricas que pueden tener nulos
cols_numericas = ["Ganados", "Goles_favor", "Puntos", "Diferencia_gol"]

# Rellenar con la mediana de cada columna
for col in cols_numericas:
    mediana = df[col].median()
    nulos = df[col].isnull().sum()
    if nulos > 0:
        df[col] = df[col].fillna(mediana)
        print(f"✅ '{col}': {nulos} nulo(s) reemplazado(s) con mediana={mediana}")

print("\n=== NULOS DESPUÉS DE CORREGIR ===")
print(df.isnull().sum())

=== NULOS ANTES DE CORREGIR ===
Posicion            0
Club                0
Partidos_jugados    0
Ganados             1
Empatados           0
Perdidos            0
Goles_favor         1
Goles_en_contra     0
Puntos              1
Diferencia_gol      1
dtype: int64
✅ 'Ganados': 1 nulo(s) reemplazado(s) con mediana=7.0
✅ 'Goles_favor': 1 nulo(s) reemplazado(s) con mediana=23.0
✅ 'Puntos': 1 nulo(s) reemplazado(s) con mediana=27.0
✅ 'Diferencia_gol': 1 nulo(s) reemplazado(s) con mediana=2.0

=== NULOS DESPUÉS DE CORREGIR ===
Posicion            0
Club                0
Partidos_jugados    0
Ganados             0
Empatados           0
Perdidos            0
Goles_favor         0
Goles_en_contra     0
Puntos              0
Diferencia_gol      0
dtype: int64


## **🔢 Paso 6 — Corrección de tipos de datos**

Cuando pandas carga un CSV, a veces interpreta números como
texto (`object`) o decimales donde deberían ser enteros.

**¿Por qué importa el tipo de dato?**
- No puedes sumar columnas de tipo `object`
- Los gráficos de barras necesitan números, no texto
- SQL rechaza comparaciones entre tipos distintos

**Tipos esperados en nuestro dataset:**

| Columna | Tipo correcto |
|---|---|
| Club | object (texto) ✅ |
| Posicion, Partidos_jugados | int64 (entero) |
| Ganados, Empatados, Perdidos | int64 (entero) |
| Goles_favor, Goles_en_contra | int64 (entero) |
| Puntos, Diferencia_gol | int64 (entero) |

> 💡 Usamos `astype(int)` para convertir. Si una columna tiene
> decimales como `20.0` los convierte a `20` sin perder información.

In [25]:
# ── Paso 6: Corrección de tipos de datos ─────────────────

print("=== TIPOS ANTES DE CORREGIR ===")
print(df.dtypes)

# Columnas que deben ser enteros
cols_enteras = ["Posicion", "Partidos_jugados", "Ganados", "Empatados",
                "Perdidos", "Goles_favor", "Goles_en_contra",
                "Puntos", "Diferencia_gol"]

for col in cols_enteras:
    df[col] = df[col].astype(int)

print("\n=== TIPOS DESPUÉS DE CORREGIR ===")
print(df.dtypes)

=== TIPOS ANTES DE CORREGIR ===
Posicion             int64
Club                object
Partidos_jugados     int64
Ganados              int64
Empatados            int64
Perdidos             int64
Goles_favor          int64
Goles_en_contra      int64
Puntos               int64
Diferencia_gol       int64
dtype: object

=== TIPOS DESPUÉS DE CORREGIR ===
Posicion             int64
Club                object
Partidos_jugados     int64
Ganados              int64
Empatados            int64
Perdidos             int64
Goles_favor          int64
Goles_en_contra      int64
Puntos               int64
Diferencia_gol       int64
dtype: object


## **💾 Paso 7 — Exportar dataset limpio**

El dataset ya está limpio. Ahora lo guardamos en la carpeta
`02_datos/procesados/` — **nunca** sobreescribimos el crudo original.

**Resumen de lo que corregimos en este notebook:**

| Problema encontrado | Solución aplicada |
|---|---|
| 3 filas duplicadas | `drop_duplicates()` |
| Variante "Millonarios F.C." | `str.replace()` |
| 4 valores nulos | `fillna(mediana)` |
| Tipos de datos incorrectos | `astype(int)` |
| Nombres con texto extra | `str.replace()` regex |

**El archivo limpio queda en:**
`Proyecto_AVD_Equipo_N3 → 02_datos → 02_Procesados → betplay_2025_limpio.csv`

> ✅ Este archivo es la entrada del siguiente notebook:
> `02_eda.ipynb` — Análisis Exploratorio de Datos.

In [26]:
# ── Paso 7: Exportar CSV limpio ───────────────────────────

ruta_salida = '/content/drive/MyDrive/PROYECTO FINAL/Proyecto-AVD-Equipo-3/02_datos/02_Procesados/betplay_2025_Limpio.csv'

df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

print("=== DATASET LIMPIO EXPORTADO ===")
print(f"Ruta: {ruta_salida}")
print(f"Filas finales:   {len(df)}")
print(f"Columnas:        {list(df.columns)}")
print(f"\nVista previa:")
print(df.head(20))

=== DATASET LIMPIO EXPORTADO ===
Ruta: /content/drive/MyDrive/PROYECTO FINAL/Proyecto-AVD-Equipo-3/02_datos/02_Procesados/betplay_2025_Limpio.csv
Filas finales:   20
Columnas:        ['Posicion', 'Club', 'Partidos_jugados', 'Ganados', 'Empatados', 'Perdidos', 'Goles_favor', 'Goles_en_contra', 'Puntos', 'Diferencia_gol']

Vista previa:
    Posicion                        Club  Partidos_jugados  Ganados  \
0         16                 Envigado FC                20        4   
1         10             Águilas Doradas                20        7   
2          1      Independiente Medellín                20       12   
3          9                  Alianza FC                20        8   
4         18           Deportivo Pereira                20        4   
5         13                 Llaneros FC                20        7   
6          2             Deportes Tolima                20       12   
7         14              Deportivo Cali                20        5   
8          6            